In [1]:
from __future__ import annotations

from collections import deque
from pathlib import Path
from typing import Dict, List, Optional, Set, Tuple
import time

Grid = List[List[int]]
Var = Tuple[int, int]
DomainMap = Dict[Var, Set[int]]

DIGITS = set(range(1, 10))
ROWS = range(9)
COLS = range(9)

In [2]:
def parse_board_lines(lines: List[str]) -> Grid:
    if len(lines) != 9:
        raise ValueError("Board file must contain exactly 9 lines.")

    board: Grid = []
    for idx, raw in enumerate(lines, start=1):
        line = raw.strip()
        if len(line) != 9 or not line.isdigit():
            raise ValueError(f"Line {idx} must contain exactly 9 digits.")
        row = [int(ch) for ch in line]
        board.append(row)
    return board


def read_board_file(path: Path) -> Grid:
    text = path.read_text(encoding="utf-8")
    lines = [ln for ln in text.splitlines() if ln.strip() != ""]
    return parse_board_lines(lines)


def write_board_file(path: Path, lines: List[str]) -> None:
    path.write_text("\n".join(lines) + "\n", encoding="utf-8")


def print_board(board: Grid, title: str = "Sudoku") -> None:
    print(f"\n{title}")
    print("=" * len(title))
    for r in ROWS:
        if r and r % 3 == 0:
            print("-" * 21)
        row_chunks: List[str] = []
        for c in COLS:
            if c and c % 3 == 0:
                row_chunks.append("|")
            v = board[r][c]
            row_chunks.append(str(v) if v != 0 else ".")
        print(" ".join(row_chunks))


def is_complete_and_valid(board: Grid) -> bool:
    expected = DIGITS
    for r in ROWS:
        if set(board[r]) != expected:
            return False
    for c in COLS:
        if {board[r][c] for r in ROWS} != expected:
            return False
    for br in range(0, 9, 3):
        for bc in range(0, 9, 3):
            box = {board[r][c] for r in range(br, br + 3) for c in range(bc, bc + 3)}
            if box != expected:
                return False
    return True

## CSP Model



In [3]:
ALL_VARS: List[Var] = [(r, c) for r in ROWS for c in COLS]


def build_peers() -> Dict[Var, Set[Var]]:
    peers: Dict[Var, Set[Var]] = {}
    for r in ROWS:
        for c in COLS:
            var = (r, c)
            p: Set[Var] = set()

            for cc in COLS:
                if cc != c:
                    p.add((r, cc))

            for rr in ROWS:
                if rr != r:
                    p.add((rr, c))

            br, bc = 3 * (r // 3), 3 * (c // 3)
            for rr in range(br, br + 3):
                for cc in range(bc, bc + 3):
                    if (rr, cc) != var:
                        p.add((rr, cc))

            peers[var] = p
    return peers


PEERS = build_peers()
ALL_ARCS: List[Tuple[Var, Var]] = [(xi, xj) for xi in ALL_VARS for xj in PEERS[xi]]


def initial_domains_from_board(board: Grid) -> DomainMap:
    domains: DomainMap = {}
    for r in ROWS:
        for c in COLS:
            v = board[r][c]
            if v == 0:
                domains[(r, c)] = set(DIGITS)
            else:
                domains[(r, c)] = {v}

    # Reject invalid clues immediately (same value in peer singletons).
    for var in ALL_VARS:
        if len(domains[var]) == 1:
            value = next(iter(domains[var]))
            for nbr in PEERS[var]:
                if len(domains[nbr]) == 1 and value == next(iter(domains[nbr])):
                    raise ValueError("Invalid puzzle: conflicting fixed clues detected.")

    return domains


def domains_to_board(domains: DomainMap) -> Grid:
    board: Grid = [[0 for _ in COLS] for _ in ROWS]
    for (r, c), d in domains.items():
        if len(d) == 1:
            board[r][c] = next(iter(d))
    return board


def clone_domains(domains: DomainMap) -> DomainMap:
    return {var: set(values) for var, values in domains.items()}

## AC-3 and Forward Checking



In [4]:
def revise(domains: DomainMap, xi: Var, xj: Var) -> bool:
    """Remove unsupported values from domain(xi) for constraint xi != xj."""
    removed = False
    dj = domains[xj]

    if len(dj) != 1:
        return False

    forbidden = next(iter(dj))
    if forbidden in domains[xi]:
        domains[xi].remove(forbidden)
        removed = True
    return removed


def ac3(domains: DomainMap, queue: Optional[deque] = None) -> bool:
    q: deque = deque(ALL_ARCS if queue is None else queue)

    while q:
        xi, xj = q.popleft()
        if revise(domains, xi, xj):
            if len(domains[xi]) == 0:
                return False
            for xk in PEERS[xi]:
                if xk != xj:
                    q.append((xk, xi))
    return True


def forward_check_and_propagate(domains: DomainMap, var: Var, value: int) -> bool:
    domains[var] = {value}

    # Forward checking: remove assigned value from all peers.
    for nbr in PEERS[var]:
        if value in domains[nbr]:
            if len(domains[nbr]) == 1:
                return False
            domains[nbr].remove(value)
            if len(domains[nbr]) == 0:
                return False

    # Enforce full arc consistency after local pruning.
    return ac3(domains)


def is_assignment_complete(domains: DomainMap) -> bool:
    return all(len(domains[var]) == 1 for var in ALL_VARS)


def select_unassigned_variable(domains: DomainMap) -> Var:
    # MRV + Degree tie-breaker over unassigned variables.
    candidates = [v for v in ALL_VARS if len(domains[v]) > 1]
    min_size = min(len(domains[v]) for v in candidates)
    mrv = [v for v in candidates if len(domains[v]) == min_size]

    if len(mrv) == 1:
        return mrv[0]

    return max(mrv, key=lambda v: sum(1 for p in PEERS[v] if len(domains[p]) > 1))


def order_domain_values(domains: DomainMap, var: Var) -> List[int]:
    return sorted(domains[var])

## Backtracking Search


In [5]:
def backtrack(domains: DomainMap, stats: Dict[str, int]) -> Optional[DomainMap]:
    stats["backtrack_calls"] += 1

    if is_assignment_complete(domains):
        return domains

    var = select_unassigned_variable(domains)

    for value in order_domain_values(domains, var):
        stats["assignments_tried"] += 1
        next_domains = clone_domains(domains)

        if not forward_check_and_propagate(next_domains, var, value):
            continue

        result = backtrack(next_domains, stats)
        if result is not None:
            return result

    stats["backtrack_failures"] += 1
    return None


def solve_sudoku_csp(board: Grid) -> Tuple[Optional[Grid], Dict[str, float]]:
    domains = initial_domains_from_board(board)
    if not ac3(domains):
        return None, {
            "backtrack_calls": 0,
            "backtrack_failures": 0,
            "assignments_tried": 0,
            "time_ms": 0.0,
        }

    stats: Dict[str, float] = {
        "backtrack_calls": 0,
        "backtrack_failures": 0,
        "assignments_tried": 0,
        "time_ms": 0.0,
    }

    t0 = time.perf_counter()
    solution_domains = backtrack(domains, stats)
    stats["time_ms"] = (time.perf_counter() - t0) * 1000

    if solution_domains is None:
        return None, stats

    solved_board = domains_to_board(solution_domains)
    return solved_board, stats

## Load the Four Boards and Solve


In [6]:
# Embedded board strings are used to create files if they are missing.
BOARD_TEXT: Dict[str, List[str]] = {
    "easy": [
        "004030050",
        "609400000",
        "005100489",
        "000060930",
        "300807002",
        "026040000",
        "453009600",
        "000004705",
        "090050200",
    ],
    "medium": [
        "000030040",
        "109700000",
        "000851070",
        "002607830",
        "906010207",
        "031502900",
        "010369000",
        "000005703",
        "090070000",
    ],
    "hard": [
        "102040007",
        "000080000",
        "009500304",
        "000607900",
        "540000026",
        "006405000",
        "708003400",
        "000010000",
        "200060509",
    ],
    "veryhard": [
        "001007000",
        "600040300",
        "000030064",
        "380076000",
        "000000036",
        "270015000",
        "000020051",
        "700100200",
        "008009000",
    ],
}


def ensure_board_files(base_dir: Path) -> None:
    for name, lines in BOARD_TEXT.items():
        path = base_dir / f"{name}.txt"
        if not path.exists():
            write_board_file(path, lines)


def load_boards(base_dir: Path) -> Dict[str, Grid]:
    ensure_board_files(base_dir)

    boards: Dict[str, Grid] = {}
    for name in ["easy", "medium", "hard", "veryhard"]:
        path = base_dir / f"{name}.txt"
        boards[name] = read_board_file(path)
    return boards


workspace = Path(".").resolve()
boards = load_boards(workspace)

results: Dict[str, Dict[str, object]] = {}

for name in ["easy", "medium", "hard", "veryhard"]:
    puzzle = boards[name]
    print_board(puzzle, f"{name.upper()} - input")

    solved, stats = solve_sudoku_csp(puzzle)
    solved_ok = solved is not None and is_complete_and_valid(solved)

    if solved is not None:
        print_board(solved, f"{name.upper()} - solved")
    else:
        print(f"\n{name.upper()} - solved")
        print("No solution found.")

    print(f"BACKTRACK calls    : {int(stats['backtrack_calls'])}")
    print(f"BACKTRACK failures : {int(stats['backtrack_failures'])}")
    print(f"Assignments tried  : {int(stats['assignments_tried'])}")
    print(f"Time (ms)          : {stats['time_ms']:.3f}")
    print(f"Solved + valid     : {solved_ok}\n")

    results[name] = {
        "solved": solved,
        "solved_ok": solved_ok,
        "stats": stats,
    }


EASY - input
. . 4 | . 3 . | . 5 .
6 . 9 | 4 . . | . . .
. . 5 | 1 . . | 4 8 9
---------------------
. . . | . 6 . | 9 3 .
3 . . | 8 . 7 | . . 2
. 2 6 | . 4 . | . . .
---------------------
4 5 3 | . . 9 | 6 . .
. . . | . . 4 | 7 . 5
. 9 . | . 5 . | 2 . .

EASY - solved
7 8 4 | 9 3 2 | 1 5 6
6 1 9 | 4 8 5 | 3 2 7
2 3 5 | 1 7 6 | 4 8 9
---------------------
5 7 8 | 2 6 1 | 9 3 4
3 4 1 | 8 9 7 | 5 6 2
9 2 6 | 5 4 3 | 8 7 1
---------------------
4 5 3 | 7 2 9 | 6 1 8
8 6 2 | 3 1 4 | 7 9 5
1 9 7 | 6 5 8 | 2 4 3
BACKTRACK calls    : 1
BACKTRACK failures : 0
Assignments tried  : 0
Time (ms)          : 0.051
Solved + valid     : True


MEDIUM - input
. . . | . 3 . | . 4 .
1 . 9 | 7 . . | . . .
. . . | 8 5 1 | . 7 .
---------------------
. . 2 | 6 . 7 | 8 3 .
9 . 6 | . 1 . | 2 . 7
. 3 1 | 5 . 2 | 9 . .
---------------------
. 1 . | 3 6 9 | . . .
. . . | . . 5 | 7 . 3
. 9 . | . 7 . | . . .

MEDIUM - solved
8 7 5 | 9 3 6 | 1 4 2
1 6 9 | 7 2 4 | 3 8 5
2 4 3 | 8 5 1 | 6 7 9
---------------------
4

In [9]:
ORDERED_BOARDS = ["easy", "medium", "hard", "veryhard"]

def board_to_lines(board: Grid) -> List[str]:
    return ["".join(str(v) for v in row) for row in board]

def board_to_pretty_text(board: Grid, title: str) -> str:
    out: List[str] = [f"\n{title}", "=" * len(title)]
    for r in ROWS:
        if r and r % 3 == 0:
            out.append("-" * 21)
        row_chunks: List[str] = []
        for c in COLS:
            if c and c % 3 == 0:
                row_chunks.append("|")
            row_chunks.append(str(board[r][c]))
        out.append(" ".join(row_chunks))
    return "\n".join(out)

# Self-contained execution: recompute if needed.
if "results" not in globals() or any(name not in results for name in ORDERED_BOARDS):
    workspace = Path(".").resolve()
    boards = load_boards(workspace)
    results = {}

    for name in ORDERED_BOARDS:
        puzzle = boards[name]
        solved, stats = solve_sudoku_csp(puzzle)
        solved_ok = solved is not None and is_complete_and_valid(solved)
        results[name] = {
            "solved": solved,
            "solved_ok": solved_ok,
            "stats": stats,
        }

print(f"{'Board':<10} {'Solved':<8} {'BT Calls':>10} {'BT Failures':>12} {'Time (ms)':>12}")
for name in ORDERED_BOARDS:
    rec = results[name]
    st = rec["stats"]
    print(
        f"{name:<10} {str(rec['solved_ok']):<8} "
        f"{int(st['backtrack_calls']):>10} "
        f"{int(st['backtrack_failures']):>12} "
        f"{st['time_ms']:>12.3f}"
    )


Board      Solved     BT Calls  BT Failures    Time (ms)
easy       True              1            0        0.051
medium     True              2            0       11.398
hard       True             13            6       22.068
veryhard   True             18           12       27.279
